In [ ]:
from google.cloud import bigquery

client = bigquery.Client()
query = """
    SELECT * FROM `numeric-advice-452700-j9.neo_bank_.perfil_usuario`
"""
df_perfil_usuario = client.query(query).to_dataframe()

df_perfil_usuario

In [ ]:
import plotly.express as px

usuarios_mcc = (
    df_perfil_usuario[df_perfil_usuario['mcc_description'].notna()]
    .groupby('mcc_description')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
    .sort_values(by='usuarios', ascending=False)
)

# Quedarse solo con los top 20 para que la gráfica sea legible
top_mcc = usuarios_mcc.head(20)

fig_mcc = px.bar(
    top_mcc,
    x='usuarios',
    y='mcc_description',
    orientation='h',
    labels={'usuarios': 'Usuarios únicos', 'mcc_description': 'Categoría MCC'},
    text='usuarios',
    color='usuarios',
    color_continuous_scale='Tealgrn'
)

fig_mcc.update_traces(textposition='outside')
fig_mcc.update_layout(
    xaxis_title='Número de usuarios',
    yaxis_title='Categoría MCC',
)

fig_mcc.show()

In [ ]:
usuarios_direction = (
    df_perfil_usuario[df_perfil_usuario['direction'].notna()]
    .groupby('direction')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
    .sort_values(by='usuarios', ascending=False)
)

# Quedarse solo con los top 20 para que la gráfica sea legible
top_direction = usuarios_direction.head(20)

fig_direction = px.bar(
    top_direction,
    x='direction',        # ahora direction en el eje X
    y='usuarios',         # usuarios en el eje Y
    labels={'usuarios': 'Usuarios únicos', 'direction': 'Direction'},
    text='usuarios',
    color='usuarios',
    color_continuous_scale='Tealgrn'
)

fig_direction.update_traces(textposition='outside')
fig_direction.update_layout(
    xaxis_title='Direction',
    yaxis_title='Número de usuarios',
)

fig_direction.show()


In [ ]:
usuarios_transactions_state = (
    df_perfil_usuario[df_perfil_usuario['transactions_state'].notna()]
    .groupby('transactions_state')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
    .sort_values(by='usuarios', ascending=False)
)

# Quedarse solo con los top 20 para que la gráfica sea legible
top_transactions_state = usuarios_transactions_state.head(20)

fig_transactions_state = px.bar(
    top_transactions_state,
    x='transactions_state',
    y='usuarios',
    labels={'usuarios': 'Usuarios únicos', 'transactions_state': 'transactions_state'},
    text='usuarios',
    color='usuarios',
    color_continuous_scale='Tealgrn'
)

fig_transactions_state.update_traces(textposition='outside')
fig_transactions_state.update_layout(
    xaxis_title='Número de usuarios',
    yaxis_title='transactions_state',
)

fig_transactions_state.show()

In [ ]:
usuarios_mcc_age = (
    df_perfil_usuario[df_perfil_usuario['mcc_description'].notna() & df_perfil_usuario['age_group'].notna()]
    .groupby(['mcc_description', 'age_group'])['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
)

usuarios_mcc_age


In [ ]:
top_mccs = (
    usuarios_mcc_age.groupby('mcc_description')['usuarios']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .index.tolist()
)

df_top_mcc_age = usuarios_mcc_age[usuarios_mcc_age['mcc_description'].isin(top_mccs)]

df_top_mcc_age

In [ ]:
fig_mcc_age = px.bar(
    df_top_mcc_age,
    x='mcc_description',
    y='usuarios',
    color='age_group',
    labels={
        'usuarios': 'Usuarios únicos',
        'mcc_description': 'Categoría MCC',
        'age_group': 'Grupo de edad'
    },
    text='usuarios',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig_mcc_age.update_traces(textposition='inside', texttemplate='%{text}')
fig_mcc_age.update_layout(
    xaxis_title='Categoría MCC',
    yaxis_title='Número de usuarios',
    barmode='stack',  # importante para que sea apilada
    xaxis_tickangle=-45
)

fig_mcc_age.show()


In [ ]:
usuarios_mcc_plan = (
    df_perfil_usuario[df_perfil_usuario['mcc_description'].notna() & df_perfil_usuario['plan'].notna()]
    .groupby(['mcc_description', 'plan'])['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
)

usuarios_mcc_plan


In [ ]:
import pandas as pd

top_mccs = (
    usuarios_mcc_plan.groupby('mcc_description')['usuarios']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .index.tolist()
)

df_top_mcc_plan = usuarios_mcc_plan[usuarios_mcc_plan['mcc_description'].isin(top_mccs)]

top_mccs, df_top_mcc_plan

In [ ]:
fig_mcc_plan = px.bar(
    df_top_mcc_plan,
    x='mcc_description',
    y='usuarios',
    color='plan',
    labels={
        'usuarios': 'Usuarios únicos',
        'mcc_description': 'Categoría MCC',
        'plan': 'Plan'
    },
    text='usuarios',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig_mcc_plan.update_traces(textposition='inside', texttemplate='%{text}')
fig_mcc_plan.update_layout(
    xaxis_title='Categoría MCC',
    yaxis_title='Número de usuarios',
    barmode='stack',
    xaxis_tickangle=-45
)

fig_mcc_plan.show()


In [ ]:
# Asegurar que haya valores válidos
df_filtrado = df_perfil_usuario[df_perfil_usuario['total_amount_usd'].notna() & df_perfil_usuario['age_group'].notna()]

df_filtrado

In [ ]:

# Paso 1: Agrupar por usuario para no duplicar montos
df_usuario = (
    df_filtrado.groupby('user_id')
    .agg({
        'age_group': 'first',
        'total_amount_usd': 'first'  # O sum si hay varias filas por usuario
    })
    .reset_index()
)
df_usuario

In [ ]:

# Paso 2: Agrupar por segmento (edad, plan, país, etc.)
monto_segmento = (
    df_usuario
    .groupby('age_group')['total_amount_usd']
    .sum()
    .reset_index()
    .sort_values(by='total_amount_usd', ascending=False)
)
monto_segmento

In [ ]:

# Visualización
fig_amount = px.bar(
    monto_segmento,
    x='age_group',
    y='total_amount_usd',
    labels={'age_group': 'age_group'.capitalize(), 'total_amount_usd': 'Monto total (USD)'},
    color='total_amount_usd',
    color_continuous_scale='Tealgrn',
    text='total_amount_usd'
)

fig_amount.update_traces(textposition='outside')
fig_amount.update_layout(
    xaxis_title='age_group'.capitalize(),
    yaxis_title='Monto total en USD'
)

In [ ]:
total_amount = df_perfil_usuario['total_amount_usd'].sum()
f"${total_amount:,.2f}"


In [ ]:
totales_por_grupo = df_perfil_usuario.groupby('age_group')['total_amount_usd'].sum()

# Formatear cada monto como string bonito
totales_formateados = totales_por_grupo.apply(lambda x: f"${x:,.2f}")
print(totales_formateados)


In [ ]:
total_amount = f"${df_perfil_usuario['total_amount_usd'].dropna().sum():,.2f}"
print(total_amount)
